In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark import StorageLevel
import mlflow
import pandas as pd

# ── Spark + MinIO ─────────────────────────────────────────────────
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("HospitalOptimizer-Prophet") \
    .config("spark.driver.memory", "4g") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://hospital-minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "hospital") \
    .config("spark.hadoop.fs.s3a.secret.key", "hospital123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# ── Charger Gold depuis MinIO ─────────────────────────────────────
df_gold = spark.read.parquet("s3a://gold/parquet/mimic_features.parquet").cache()
print(f"✅ Gold chargé : {df_gold.count():,} lignes")

✅ Gold chargé : 122,756 lignes


In [2]:
# Prophet attend deux colonnes : ds (date) et y (valeur)
# On agrège le nombre d'admissions par jour

admissions_daily = df_gold \
    .withColumn("ds", F.to_date("ADMITTIME")) \
    .groupBy("ds") \
    .agg(F.count("HADM_ID").alias("y")) \
    .orderBy("ds") \
    .filter(F.col("ds").isNotNull())

# Convertir en Pandas pour Prophet
df_prophet = admissions_daily.toPandas()

print(f"✅ Données préparées pour Prophet")
print(f"   Période : {df_prophet['ds'].min()} → {df_prophet['ds'].max()}")
print(f"   Jours   : {len(df_prophet):,}")
print(f"\nAperçu news:")
print(df_prophet.head(5))

✅ Données préparées pour Prophet
   Période : 2100-01-01 → 2210-12-30
   Jours   : 34,926

Aperçu news:
           ds  y
0  2100-01-01  6
1  2100-01-02  5
2  2100-01-03  3
3  2100-01-04  6
4  2100-01-05  3


In [3]:
from prophet import Prophet
from sklearn.metrics import mean_absolute_error
import numpy as np

# ── Split train/test ──────────────────────────────────────────────
split_idx = int(len(df_prophet) * 0.80)
train     = df_prophet[:split_idx]
test      = df_prophet[split_idx:]

print(f"Train : {len(train):,} jours")
print(f"Test  : {len(test):,} jours")

# ── Entraîner Prophet ─────────────────────────────────────────────
model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    changepoint_prior_scale=0.05
)

model.fit(train)
print("✅ Prophet entraîné")

Importing plotly failed. Interactive plots will not work.


Train : 27,940 jours
Test  : 6,986 jours


10:34:24 - cmdstanpy - INFO - Chain [1] start processing
10:34:28 - cmdstanpy - INFO - Chain [1] done processing


✅ Prophet entraîné


In [4]:
# Prédiction et évaluation

# ── Prédire sur la période de test ───────────────────────────────
future = model.make_future_dataframe(periods=len(test), freq='D')
forecast = model.predict(future)

# ── Extraire les prédictions sur la période de test ───────────────
forecast_test = forecast.tail(len(test))[['ds', 'yhat', 'yhat_lower', 'yhat_upper']]
forecast_test = forecast_test.reset_index(drop=True)
test          = test.reset_index(drop=True)

# ── Calculer le MAE ───────────────────────────────────────────────
mae = mean_absolute_error(test['y'], forecast_test['yhat'])

print("=" * 50)
print("RÉSULTATS PROPHET")
print("=" * 50)
print(f"MAE         : {mae:.2f} admissions")
print(f"Cible       : < 2h (en nombre d'admissions)")
print(f"\nAperçu prédictions vs réalité :")
comparison = pd.DataFrame({
    'date':     test['ds'],
    'réel':     test['y'],
    'prédit':   forecast_test['yhat'].round(1),
    'erreur':   (test['y'] - forecast_test['yhat']).abs().round(1)
})
print(comparison.head(10).to_string(index=False))

RÉSULTATS PROPHET
MAE         : 1.82 admissions
Cible       : < 2h (en nombre d'admissions)

Aperçu prédictions vs réalité :
      date  réel  prédit  erreur
2188-09-29     4     3.5     0.5
2188-09-30     2     3.5     1.5
2188-10-01     2     3.5     1.5
2188-10-02     3     3.4     0.4
2188-10-05     5     3.5     1.5
2188-10-07     5     3.5     1.5
2188-10-09     1     3.5     2.5
2188-10-10     1     3.5     2.5
2188-10-11     5     3.5     1.5
2188-10-12     1     3.5     2.5


In [13]:
import mlflow

mlflow.set_tracking_uri("http://hospital-mlflow:5000")
mlflow.set_experiment("/hospital-optimizer/prophet")
print("✅ MLflow connecté maintenant")

✅ MLflow connecté maintenant


In [17]:
# Logger dans mlflow

import mlflow.prophet
import os

os.makedirs("/home/jovyan/mlflow-artifacts", exist_ok=True)
mlflow.set_tracking_uri("http://hospital-mlflow:5000")
mlflow.set_experiment("/hospital-optimizer/prophet")

with mlflow.start_run(run_name="prophet_admissions_v1"):

    mlflow.log_param("yearly_seasonality", True)
    mlflow.log_param("weekly_seasonality", True)
    mlflow.log_param("changepoint_prior_scale", 0.05)
    mlflow.log_param("train_size", len(train))
    mlflow.log_param("test_size", len(test))

    mlflow.log_metric("mae", round(mae, 2))
    mlflow.log_metric("mae_target", 2.0)
    mlflow.log_metric("kpi_achieved", 1 if mae < 2 else 0)

    # Sauvegarder seulement les métriques et params sans le modèle
    print("✅ Run MLflow enregistré")
    print(f"   MAE : {mae:.2f}")
    print(f"   KPI : {'✅ ATTEINT' if mae < 2 else '❌ NON ATTEINT'}")

# Sauvegarder le modèle localement séparément
import pickle
with open("/home/jovyan/src/models/prophet_model.pkl", "wb") as f:
    pickle.dump(model, f)
print("✅ Modèle sauvegardé localement")

✅ Run MLflow enregistré
   MAE : 1.82
   KPI : ✅ ATTEINT
✅ Modèle sauvegardé localement
